# JAX Eccentric Model with pyTDI

This notebook shows the standalone `eGB-multi` package workflow:

1. Build an eccentric compact-binary source.
2. Compare the switchable source-physics modes.
3. Generate JAX six-link responses.
4. Route the links through pyTDI to Michelson `X/Y/Z` and convert to `A/E/T`.

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from egb_jax_eccentric import (
    EccentricBinaryParams,
    aet_from_xyz,
    eccentric_complex_strain,
    eccentric_links_jax,
    eccentric_xyz_jax,
    lisa_orbit,
    precompute_jax_link_geometry,
)

plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True})

## Source and LISA State

The internal `lisa_orbit` helper gives a lightweight analytic LISA orbit. For production comparisons, replace it with `state_from_lisaorbits(default_lisaorbits(...), times)`.

In [ ]:
duration = 20_000.0
samples = 1024
times = np.linspace(0.0, duration, samples, endpoint=False)
state = lisa_orbit(times)
geometry = precompute_jax_link_geometry(state)

source = EccentricBinaryParams(
    mean_motion=np.pi * 1.0e-4,
    eccentricity=0.08,
    m1_solar=0.6,
    m2_solar=0.4,
    beta=0.2,
    lambda_=0.4,
    psi=0.3,
    inclination=0.8,
    phi0=0.2,
)

source

## Switchable Source Physics

`physics_mode` can be `newtonian`, `1pn_no_periastron`, or `1pn`. The JAX exact-link path accepts the same switch.

In [ ]:
physics_modes = ["newtonian", "1pn_no_periastron", "1pn"]
strains = {
    mode: eccentric_complex_strain(source, times, physics_mode=mode)
    for mode in physics_modes
}

for mode, strain in strains.items():
    plt.plot(times / 3600.0, np.real(strain), label=mode)
plt.xlabel("time [hours]")
plt.ylabel("Re[h+ - i hx]")
plt.legend()
plt.title("Switchable eccentric source model")
plt.show()

## JAX Links and pyTDI `X/Y/Z`

`eccentric_links_jax` returns six pyTDI-labelled one-way GW links. `eccentric_xyz_jax` is the convenience path that immediately routes those links through pyTDI.

In [ ]:
links = eccentric_links_jax(source, geometry, batch_size=1, physics_mode="1pn")
print({label: value.shape for label, value in links.items()})

xyz = eccentric_xyz_jax(
    state,
    source,
    geometry=geometry,
    batch_size=1,
    physics_mode="1pn",
    measurement_order=3,
    delay_order=3,
)
print({channel: value.shape for channel, value in xyz.items()})

In [ ]:
for channel, series in xyz.items():
    plt.plot(times / 3600.0, np.real(series), label=channel)
plt.xlabel("time [hours]")
plt.ylabel("Re[channel]")
plt.legend()
plt.title("pyTDI Michelson X/Y/Z from JAX eccentric links")
plt.show()

## `A/E/T` Conversion

In [ ]:
aet = aet_from_xyz(xyz)

for channel, series in aet.items():
    rms = np.sqrt(np.mean(np.abs(series) ** 2))
    print(f"{channel}: rms={rms:.6e}")

for channel, series in aet.items():
    plt.plot(times / 3600.0, np.real(series), label=channel)
plt.xlabel("time [hours]")
plt.ylabel("Re[channel]")
plt.legend()
plt.title("Orthogonal A/E/T channels")
plt.show()